# Movie Review Sentiment Analysis
### End-to-End NLP & Machine Learning Pipeline
**Target Role**: Tata Power Graduate Engineer Trainee (AI/ML)

---
### Project Objective
Develop a robust, lightweight, and completely explainable Natural Language Processing (NLP) system to classify movie reviews into **Positive** or **Negative** sentiment.

### Pipeline Overview
1. **Data Ingestion**: Load 50,000 raw reviews from IMDB dataset.
2. **Data Cleaning & Inspection**: Check missing values, deduplicate records, inspect class distribution.
3. **Text Preprocessing**: HTML tag removal, contraction expansion (`didn't` -> `did not`), lowercasing, and punctuation filtering.
4. **Exploratory Data Analysis (EDA)**: Class balance and review length distributions.
5. **Stratified Train/Test Split**: 80/20 train/test split to prevent data leakage.
6. **TF-IDF Feature Extraction**: Unigrams and bigrams (`ngram_range=(1,2)`), vocabulary size of 10,000, sublinear TF.
7. **Model Training & Comparison**: Logistic Regression vs. Multinomial Naive Bayes.
8. **Evaluation Metrics**: Accuracy, Precision, Recall, F1-Score, and Confusion Matrices.
9. **Model Serialization**: Saving best model and vectorizer with `joblib`.

In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is available for imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_preprocessing import clean_text, load_and_preprocess_data
from src.predict import SentimentPredictor

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries successfully imported.")

## 1. Load and Inspect Raw Dataset

In [ ]:
raw_data_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'imdb_reviews.csv')
df_raw = pd.read_csv(raw_data_path)

print(f"Total raw records: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")
print(f"Missing values:\n{df_raw.isnull().sum()}")
print(f"Duplicate reviews count: {df_raw.duplicated(subset=['review']).sum()}")
print(f"\nClass Balance:\n{df_raw['sentiment'].value_counts()}")

df_raw.head(3)

## 2. Text Preprocessing & Negation Handling

In [ ]:
sample_raw = "I didn't like this movie at all! It's terrible <br /><br /> See http://movie.com"
sample_cleaned = clean_text(sample_raw)

print("--- Preprocessing Demonstration ---")
print(f"Raw:     {sample_raw}")
print(f"Cleaned: {sample_cleaned}")

## 3. Load Model Evaluation Summary & Metadata

In [ ]:
metadata_path = os.path.join(PROJECT_ROOT, 'models', 'model_metadata.json')
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f"Winning Model: {metadata['best_model']}")
metrics_df = pd.DataFrame(metadata['metrics_summary']).T
metrics_df[['accuracy', 'precision', 'recall', 'f1_score']]

## 4. Live Prediction with Pretrained Pipeline

In [ ]:
predictor = SentimentPredictor(models_dir=os.path.join(PROJECT_ROOT, 'models'))

test_samples = [
    "This was an extraordinary film with stellar performances by all actors!",
    "Terrible plot, boring pacing, and completely unconvincing acting.",
    "I hoped it would be decent, but it was not good."
]

for text in test_samples:
    res = predictor.predict(text)
    print(f"Input:       \"{text}\"")
    print(f"Prediction:  {res['prediction']} (Confidence: {res['confidence']*100:.2f}%)")
    print(f"Probabilities: [Positive: {res['positive_probability']:.3f} | Negative: {res['negative_probability']:.3f}]")
    print("-" * 60)